# Xây Dựng & Chuẩn Hóa Bảng Cầu Nối: `job_skills`

Notebook này tập trung xây dựng bảng cầu nối quan hệ Nhiều-Nhiều (Many-to-Many Bridge Table) **`job_skills`** giữa bảng `jobs` và bảng `skills`:
- Mỗi bản ghi đại diện cho một kỹ năng được yêu cầu hoặc ưu tiên cho một vị trí tuyển dụng cụ thể.
- Hỗ trợ đa phương pháp trích xuất: từ thẻ tag trực tiếp (`tag_direct`), từ từ khóa chức danh (`title_keyword`), từ yêu cầu công việc (`requirement_rule`), và từ mô tả công việc (`description_rule`).
- Tính toán điểm tin cậy (`confidence`) từ 0.800 đến 1.000 tương ứng với từng nguồn trích xuất.
- Phân loại kỹ năng bắt buộc (`is_required = True`) và kỹ năng điểm cộng/ưu tiên (`is_required = False`).
- Lưu trữ đoạn trích bằng chứng thực tế (`evidence_text`) giúp dễ dàng kiểm chứng và giải thích dữ liệu.
- Kiểm tra nghiêm ngặt toàn vẹn khóa ngoại (FK) với `jobs` và `skills`, đảm bảo tính duy nhất của khóa chính phức hợp `(job_id, skill_id)`.

In [1]:
import os
import re
import json
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Môi trường xử lý bảng job_skills đã sẵn sàng!')

Môi trường xử lý bảng job_skills đã sẵn sàng!


## 1. Nạp Dữ Liệu Bronze & Danh Mục `skills` Chuẩn Hóa

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
skills_nb_path = os.path.abspath('04_skills.ipynb')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

with open(skills_nb_path, 'r', encoding='utf-8') as f:
    nb4 = json.load(f)
scope4 = {}
exec(''.join(nb4['cells'][3]['source']), scope4)
skills_def = scope4['skills_raw_definition']

skill_lookup = {}
for idx, s in enumerate(skills_def):
    s_id = idx + 1
    skill_lookup[s['name']] = s_id

print(f'Tổng số tin tuyển dụng nạp vào: {len(bronze_listings):,} tin')
print(f'Tổng số kỹ năng tham chiếu từ bảng skills: {len(skill_lookup):,} kỹ năng')

Tổng số kỹ năng được định nghĩa trong danh mục: 76 kỹ năng
Tổng số nhóm chuyên môn (categories): 10 nhóm
Tổng số tin tuyển dụng nạp vào: 2,327 tin
Tổng số kỹ năng tham chiếu từ bảng skills: 76 kỹ năng


## 2. Thiết Kế Pipeline Trích Xuất Kỹ Năng Đa Tầng (Multi-Layer Extraction Pipeline)

Thiết kế bộ đối sánh biểu thức chính quy (Regex Matchers) tối ưu kèm theo logic xếp hạng mức độ ưu tiên:
1. `tag_direct` (Confidence: 1.000): Trích từ trường `skills_raw` của trang danh mục.
2. `title_keyword` (Confidence: 0.950): Trích từ chức danh công việc trong tiêu đề `title_raw`.
3. `requirement_rule` (Confidence: 0.900): Trích từ phần yêu cầu ứng viên `requirements_raw`.
4. `description_rule` (Confidence: 0.800): Trích từ phần mô tả trách nhiệm công việc `responsibilities_raw`.
5. Phân loại bắt buộc (`is_required`): Gán False nếu ngữ cảnh chứa 'ưu tiên', 'điểm cộng', 'preferred', 'nice to have'.

In [3]:
skill_matchers = []
for idx, s in enumerate(skills_def):
    skill_id = idx + 1
    name = s['name']
    
    if name == 'Java':
        pat = r'\bjava\b(?!script)'
    elif name == 'JavaScript':
        pat = r'\bjavascript\b|\bjs\b'
    elif name == 'TypeScript':
        pat = r'\btypescript\b|\bts\b'
    elif name == 'C++':
        pat = r'\bc\+\+\b|\bcpp\b'
    elif name == 'C#':
        pat = r'\bc#\b|\bcsharp\b'
    elif name == 'Golang':
        pat = r'\bgolang\b|\bgo\b'
    elif name == '.NET':
        pat = r'\b\.net\b|\basp\.net\b'
    elif name == 'Git':
        pat = r'\bgit\b(?!lab|hub)'
    elif name == 'React':
        pat = r'\breact\b(?! native)|\breactjs\b'
    elif name == 'Vue.js':
        pat = r'\bvue\b|\bvuejs\b|\bvue\.js\b'
    elif name == 'Node.js':
        pat = r'\bnode\b|\bnodejs\b|\bnode\.js\b'
    elif name == 'Next.js':
        pat = r'\bnextjs\b|\bnext\.js\b'
    elif name == 'Nuxt.js':
        pat = r'\bnuxtjs\b|\bnuxt\.js\b'
    elif name == 'Express.js':
        pat = r'\bexpressjs\b|\bexpress\.js\b'
    elif name == 'PostgreSQL':
        pat = r'\bpostgres\b|\bpostgresql\b'
    elif name == 'Microsoft SQL Server':
        pat = r'\bsql server\b|\bmssql\b'
    elif name == 'Oracle Database':
        pat = r'\boracle\b'
    elif name == 'Google Cloud (GCP)':
        pat = r'\bgcp\b|\bgoogle cloud\b'
    elif name == 'Kubernetes':
        pat = r'\bkubernetes\b|\bk8s\b'
    elif name == 'CI/CD':
        pat = r'\bci/cd\b|\bcicd\b'
    elif name == 'LLM (Generative AI)':
        pat = r'\bllm\b|\bgenerative ai\b|\brag\b|\blarge language\b'
    elif name == 'iOS Development':
        pat = r'\bios\b'
    elif name == 'Android Development':
        pat = r'\bandroid\b'
    elif name == 'Manual Testing':
        pat = r'\bmanual test\b|\bmanual testing\b'
    elif name == 'Automation Testing':
        pat = r'\bautomation test\b|\bautomation testing\b|\bauto test\b'
    elif name == 'HTML/CSS':
        pat = r'\bhtml\b|\bcss\b'
    elif name == 'Tailwind CSS':
        pat = r'\btailwind\b|\btailwindcss\b'
    elif name == 'SQL':
        pat = r'\bsql\b'
    else:
        pat = r'\b' + re.escape(name.lower()) + r'\b'
        
    skill_matchers.append({
        'skill_id': skill_id,
        'skill_name': name,
        'regex': re.compile(pat, re.IGNORECASE)
    })

def check_is_preferred(context_text):
    if not context_text:
        return False
    s = str(context_text).lower()
    pref_terms = ['ưu tiên', 'uu tien', 'lợi thế', 'loi the', 'plus', 'nice to have', 'preferred', 'điểm cộng']
    return any(term in s for term in pref_terms)

print('Bộ matcher và hàm nhận diện ngữ cảnh đã được khởi tạo thành công!')

Bộ matcher và hàm nhận diện ngữ cảnh đã được khởi tạo thành công!


## 3. Thực Thi Trích Xuất & Khởi Tạo Bảng `job_skills` Hoàn Chỉnh

In [4]:
job_skills_dict = {}
now_iso = datetime.now().isoformat()

for idx in range(len(bronze_listings)):
    job_id = idx + 1
    l = bronze_listings[idx]
    d = bronze_details[idx]
    
    raw_tags = l.get('skills_raw') or []
    if isinstance(raw_tags, str):
        raw_tags = [x.strip() for x in raw_tags.split(',') if x.strip()]
    tag_str = ' '.join(raw_tags)
    
    title_raw = str(l.get('title_raw', '') or '')
    req_raw = str(d.get('requirements_raw', '') or '')
    resp_raw = str(d.get('responsibilities_raw', '') or '')
    
    for sm in skill_matchers:
        s_id = sm['skill_id']
        reg = sm['regex']
        key = (job_id, s_id)
        
        # 1. Thứ tự ưu tiên 1: tag_direct (Confidence: 1.000)
        m_tag = reg.search(tag_str)
        if m_tag:
            ev = f"Tag: {tag_str[max(0, m_tag.start()-10):min(len(tag_str), m_tag.end()+25)].strip()}"
            job_skills_dict[key] = {
                'job_id': job_id,
                'skill_id': s_id,
                'is_required': True,
                'confidence': 1.000,
                'extraction_method': 'tag_direct',
                'evidence_text': ev[:255],
                'created_at': now_iso
            }
            continue
            
        # 2. Thứ tự ưu tiên 2: title_keyword (Confidence: 0.950)
        m_title = reg.search(title_raw)
        if m_title:
            ev = f"Tiêu đề: {title_raw[:120].strip()}"
            job_skills_dict[key] = {
                'job_id': job_id,
                'skill_id': s_id,
                'is_required': True,
                'confidence': 0.950,
                'extraction_method': 'title_keyword',
                'evidence_text': ev[:255],
                'created_at': now_iso
            }
            continue
            
        # 3. Thứ tự ưu tiên 3: requirement_rule (Confidence: 0.900)
        m_req = reg.search(req_raw)
        if m_req:
            start_p = max(0, m_req.start() - 30)
            end_p = min(len(req_raw), m_req.end() + 60)
            snippet = req_raw[start_p:end_p].replace('\n', ' ').strip()
            is_pref = check_is_preferred(snippet)
            ev = f"Yêu cầu: ...{snippet}..."
            job_skills_dict[key] = {
                'job_id': job_id,
                'skill_id': s_id,
                'is_required': not is_pref,
                'confidence': 0.900,
                'extraction_method': 'requirement_rule',
                'evidence_text': ev[:255],
                'created_at': now_iso
            }
            continue
            
        # 4. Thứ tự ưu tiên 4: description_rule (Confidence: 0.800)
        m_resp = reg.search(resp_raw)
        if m_resp:
            start_p = max(0, m_resp.start() - 30)
            end_p = min(len(resp_raw), m_resp.end() + 60)
            snippet = resp_raw[start_p:end_p].replace('\n', ' ').strip()
            ev = f"Mô tả: ...{snippet}..."
            job_skills_dict[key] = {
                'job_id': job_id,
                'skill_id': s_id,
                'is_required': True,
                'confidence': 0.800,
                'extraction_method': 'description_rule',
                'evidence_text': ev[:255],
                'created_at': now_iso
            }
            continue

df_job_skills = pd.DataFrame(list(job_skills_dict.values()))

print('Khởi tạo bảng job_skills thành công:')
print(f'  - Tổng số cặp liên kết (job_id, skill_id): {len(df_job_skills):,}')
print(f'  - Số lượng tin có ít nhất 1 kỹ năng: {df_job_skills["job_id"].nunique():,} / {len(bronze_listings)} ({df_job_skills["job_id"].nunique()/len(bronze_listings)*100:.1f}%)')
print(f'  - Tính duy nhất của Primary Key (job_id, skill_id): {len(df_job_skills) == len(df_job_skills.drop_duplicates(subset=["job_id", "skill_id"]))}')
print(f'  - Toàn vẹn khóa ngoại job_id: {df_job_skills["job_id"].between(1, len(bronze_listings)).all()} (100.0% hợp lệ)')
print(f'  - Toàn vẹn khóa ngoại skill_id: {df_job_skills["skill_id"].between(1, len(skills_def)).all()} (100.0% hợp lệ)')

Khởi tạo bảng job_skills thành công:
  - Tổng số cặp liên kết (job_id, skill_id): 4,938
  - Số lượng tin có ít nhất 1 kỹ năng: 1,253 / 2327 (53.8%)
  - Tính duy nhất của Primary Key (job_id, skill_id): True
  - Toàn vẹn khóa ngoại job_id: True (100.0% hợp lệ)
  - Toàn vẹn khóa ngoại skill_id: True (100.0% hợp lệ)


## 4. Phân Tích Thống Kê, Đánh Giá Chất Lượng & Xem Trước Bảng `job_skills`

In [5]:
print('--- 1. CẤU TRÚC BẢNG job_skills (INFO) ---')
print(df_job_skills.info())

print('\n--- 2. PHÂN BỐ THEO PHƯƠNG PHÁP TRÍCH XUẤT (extraction_method) ---')
method_counts = df_job_skills['extraction_method'].value_counts()
method_percent = df_job_skills['extraction_method'].value_counts(normalize=True) * 100
df_method = pd.DataFrame({'Số cặp liên kết': method_counts, 'Tỷ lệ (%)': method_percent.round(2)})
print(df_method.to_string())

print('\n--- 3. PHÂN BỐ TÍNH CHẤT KỸ NĂNG (is_required) ---')
req_counts = df_job_skills['is_required'].value_counts()
req_percent = df_job_skills['is_required'].value_counts(normalize=True) * 100
df_req = pd.DataFrame({'Số lượng': req_counts, 'Tỷ lệ (%)': req_percent.round(2)})
print(df_req.to_string())

print('\n--- 4. TOP 15 KỸ NĂNG ĐƯỢC TUYỂN DỤNG NHIỀU NHẤT (SỐ LƯỢNG JOB CẦN) ---')
id_to_name = {idx + 1: s['name'] for idx, s in enumerate(skills_def)}
id_to_cat = {idx + 1: s['cat'] for idx, s in enumerate(skills_def)}
top_skills = df_job_skills['skill_id'].value_counts().head(15)
df_top = pd.DataFrame({
    'skill_id': top_skills.index,
    'Tên kỹ năng': [id_to_name[i] for i in top_skills.index],
    'Nhóm chuyên môn': [id_to_cat[i] for i in top_skills.index],
    'Số lượng job': top_skills.values,
    'Tỷ lệ (%)': (top_skills.values / len(bronze_listings) * 100).round(2)
})
print(df_top.to_string(index=False))

print('\n--- 5. PREVIEW 10 DÒNG BẢNG job_skills (CHUẨN POSTGRESQL) ---')
preview_cols = ['job_id', 'skill_id', 'is_required', 'confidence', 'extraction_method', 'evidence_text']
print(df_job_skills[preview_cols].head(10).to_string(index=False))

--- 1. CẤU TRÚC BẢNG job_skills (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 4938 entries, 0 to 4937
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_id             4938 non-null   int64  
 1   skill_id           4938 non-null   int64  
 2   is_required        4938 non-null   bool   
 3   confidence         4938 non-null   float64
 4   extraction_method  4938 non-null   str    
 5   evidence_text      4938 non-null   str    
 6   created_at         4938 non-null   str    
dtypes: bool(1), float64(1), int64(2), str(3)
memory usage: 939.5 KB
None

--- 2. PHÂN BỐ THEO PHƯƠNG PHÁP TRÍCH XUẤT (extraction_method) ---
                   Số cặp liên kết  Tỷ lệ (%)
extraction_method                            
requirement_rule              3426      69.38
tag_direct                     952      19.28
description_rule               386       7.82
title_keyword                  174       3.52

--- 3. PHÂN